# 10 EDA — Final Summary

Consolidated structural findings from the nine raw Olist tables. No source data is modified.

## Dataset overview

Load all nine raw CSVs with pandas for cross-table validation.

In [1]:
import pandas as pd
from pathlib import Path
raw_path = Path('../data/raw/olist')
files = sorted(raw_path.glob('*.csv'))
tables = {path.stem: pd.read_csv(path) for path in files}
len(tables)

9

**Observation:** Nine raw tables are available.

## Total tables, rows, and columns

Create the table inventory.

In [2]:
inventory = pd.DataFrame([{'table': name, 'rows': len(df), 'columns': len(df.columns)} for name, df in tables.items()]).sort_values('table').reset_index(drop=True)
inventory

                               table     rows  columns
0            olist_customers_dataset    99441        5
1          olist_geolocation_dataset  1000163        5
2          olist_order_items_dataset   112650        7
3       olist_order_payments_dataset   103886        5
4        olist_order_reviews_dataset    99224        7
5               olist_orders_dataset    99441        8
6             olist_products_dataset    32951        9
7              olist_sellers_dataset     3095        4
8  product_category_name_translation       71        2

**Observation:** The inventory records the grain and scale of every source table.

## Missing value summary

Summarize missing cells by table.

In [3]:
missing_summary = pd.DataFrame([{'table': name, 'missing_cells': int(df.isna().sum().sum()), 'columns_with_missing': int((df.isna().sum() > 0).sum())} for name, df in tables.items()]).sort_values('table').reset_index(drop=True)
missing_summary

                               table  missing_cells  columns_with_missing
0            olist_customers_dataset              0                     0
1          olist_geolocation_dataset              0                     0
2          olist_order_items_dataset              0                     0
3       olist_order_payments_dataset              0                     0
4        olist_order_reviews_dataset         145903                     2
5               olist_orders_dataset           4908                     3
6             olist_products_dataset           2448                     8
7              olist_sellers_dataset              0                     0
8  product_category_name_translation              0                     0

**Observation:** Missingness is concentrated in orders, products, and reviews.

## Duplicate summary

Count exact duplicate rows.

In [4]:
duplicate_summary = pd.DataFrame([{'table': name, 'duplicate_rows': int(df.duplicated().sum())} for name, df in tables.items()]).sort_values('table').reset_index(drop=True)
duplicate_summary

                               table  duplicate_rows
0            olist_customers_dataset               0
1          olist_geolocation_dataset          261831
2          olist_order_items_dataset               0
3       olist_order_payments_dataset               0
4        olist_order_reviews_dataset               0
5               olist_orders_dataset               0
6             olist_products_dataset               0
7              olist_sellers_dataset               0
8  product_category_name_translation               0

**Observation:** Only geolocation contains exact duplicate rows.

## Primary key validation summary

Validate documented candidate keys.

In [5]:
pk_map = {'olist_customers_dataset': ['customer_id'], 'olist_orders_dataset': ['order_id'], 'olist_order_items_dataset': ['order_id', 'order_item_id'], 'olist_order_payments_dataset': ['order_id', 'payment_sequential'], 'olist_order_reviews_dataset': ['review_id', 'order_id'], 'olist_products_dataset': ['product_id'], 'olist_sellers_dataset': ['seller_id'], 'product_category_name_translation': ['product_category_name']}
pk_summary = pd.DataFrame([{'table': name, 'primary_key': ', '.join(cols), 'duplicate_key_rows': int(tables[name].duplicated(subset=cols, keep=False).sum()), 'blank_key_rows': int(tables[name][cols].isna().any(axis=1).sum()), 'valid': not tables[name].duplicated(subset=cols, keep=False).any() and not tables[name][cols].isna().any(axis=1).any()} for name, cols in pk_map.items()])
pk_summary

                               table  ... valid
0            olist_customers_dataset  ...  True
1               olist_orders_dataset  ...  True
2          olist_order_items_dataset  ...  True
3       olist_order_payments_dataset  ...  True
4        olist_order_reviews_dataset  ...  True
5             olist_products_dataset  ...  True
6              olist_sellers_dataset  ...  True
7  product_category_name_translation  ...  True

[8 rows x 5 columns]

**Observation:** All eight documented candidate keys are unique and nonblank; geolocation has no raw natural key.

## Foreign key validation summary

Measure orphan child rows for core and lookup relationships.

In [6]:
relationships = [('orders.customer_id','olist_orders_dataset','customer_id','olist_customers_dataset','customer_id'),('items.order_id','olist_order_items_dataset','order_id','olist_orders_dataset','order_id'),('items.product_id','olist_order_items_dataset','product_id','olist_products_dataset','product_id'),('items.seller_id','olist_order_items_dataset','seller_id','olist_sellers_dataset','seller_id'),('payments.order_id','olist_order_payments_dataset','order_id','olist_orders_dataset','order_id'),('reviews.order_id','olist_order_reviews_dataset','order_id','olist_orders_dataset','order_id'),('products.category','olist_products_dataset','product_category_name','product_category_name_translation','product_category_name'),('customers.zip','olist_customers_dataset','customer_zip_code_prefix','olist_geolocation_dataset','geolocation_zip_code_prefix'),('sellers.zip','olist_sellers_dataset','seller_zip_code_prefix','olist_geolocation_dataset','geolocation_zip_code_prefix')]
fk_rows = []
for label, child, child_col, parent, parent_col in relationships:
    values = tables[child][child_col]
    parent_values = set(tables[parent][parent_col])
    checked = values.notna()
    fk_rows.append({'relationship': label, 'orphan_rows': int((checked & ~values.isin(parent_values)).sum())})
fk_summary = pd.DataFrame(fk_rows)
fk_summary

         relationship  orphan_rows
0  orders.customer_id            0
1      items.order_id            0
2    items.product_id            0
3     items.seller_id            0
4   payments.order_id            0
5    reviews.order_id            0
6   products.category           13
7       customers.zip          278
8         sellers.zip            7

**Observation:** Six core transaction relationships have zero orphans; category and geographic lookup coverage is incomplete.

## Data quality issues discovered

1. Orders have 4,908 missing lifecycle timestamps.
2. Reviews have 145,903 missing optional text cells.
3. Products have 2,448 missing metadata or measurement cells.
4. Geolocation has 261,831 exact duplicate rows and no natural raw key.
5. Two categories covering 13 products lack translations.
6. Geolocation lacks matches for 278 customer rows and 7 seller rows.
7. Reviews require a composite key; neither identifier alone is unique.
8. Some orders lack item, payment, or review child records.
9. Two product headers contain the source spelling `lenght`.

## Risks before ETL

Direct geolocation joins can multiply facts; silent null handling can misstate fulfilment or review coverage; single-column review keys can lose records; and incomplete lookups can drop products, customers, or sellers from reporting.

## Cleaning strategy recommendation

Preserve raw files, define typed staging tables, convert timestamps with validation, retain meaningful nulls, quarantine invalid rows, deduplicate geolocation under an approved rule, and map unknown lookups to explicit controlled values rather than dropping facts.

## Recommended ETL plan

1. Validate file manifest and schemas.
2. Load immutable raw copies into staging.
3. Parse types and timestamps with rejection logging.
4. enforce verified keys.
5. Build controlled category and ZIP-prefix dimensions.
6. Load customers, products, sellers, and dimensions.
7. Load orders, items, payments, and reviews.
8. Run row-count, key, FK, and reconciliation tests.
9. Publish analytics-ready views.